In [ ]:
# @title 1. Environment Setup & Clone Source Code
import os
import sys
from google.colab import drive, userdata

drive.mount('/content/drive')

!git clone https://github.com/Tommachilez/improving-learned-index.git
!git clone https://github.com/tommachilez/virag4fc.git

!pip install gradio google-generativeai rank_bm25 torch transformers datasets underthesea py_vncorenlp

api_key = userdata.get('GEMINI_1.5')

print("✅ Môi trường đã sẵn sàng!")

Mounted at /content/drive
Cloning into 'improving-learned-index'...
remote: Enumerating objects: 1271, done.
remote: Counting objects: 100% (367/367), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1271 (delta 318), reused 305 (delta 278), pack-reused 904 (from 1)
Receiving objects: 100% (1271/1271), 211.77 KiB | 7.84 MiB/s, done.
Resolving deltas: 100% (930/930), done.
Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 673, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 673 (delta 227), reused 175 (delta 94), pack-reused 344 (from 1)
Receiving objects: 100% (673/673), 5.40 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (451/451), done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# @title 2. Load DeeperImpact & BM25 (Optimized with Pre-tokenized Data)
import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm

DATASET_PATH = "/content/drive/MyDrive/KLTN_Project/Datasets"

DOC_MAPPING_CSV = f"{DATASET_PATH}/unique_doc_mapping.csv"

PRETOKENIZED_PATH = f"{DATASET_PATH}/vn_mining/corpus_pretokenized.jsonl"

INVERTED_INDEX_PATH = f"{DATASET_PATH}/deeperimpact_vifc/inverted_indices"
CHECKPOINT_PATH = "/content/drive/MyDrive/KLTN_Project/Models/deeperimpact_xlmr_vifc/DeepImpact_final.pt"

print("⏳ Đang tải dữ liệu...")

print(f"   -> Loading Raw Text mapping từ {os.path.basename(DOC_MAPPING_CSV)}...")
try:
    df_docs = pd.read_csv(DOC_MAPPING_CSV)
    doc_map = dict(zip(df_docs['doc_id'].astype(str), df_docs['document'].astype(str)))
    print(f"      ✅ Loaded {len(doc_map)} docs for display.")
except Exception as e:
    print(f"⚠️ Lỗi load CSV Mapping: {e}")
    doc_map = {}

print(f"   -> Loading Pre-tokenized Data từ {os.path.basename(PRETOKENIZED_PATH)}...")
tokenized_corpus = []
corpus_ids = []

try:
    with open(PRETOKENIZED_PATH, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading JSONL"):
            item = json.loads(line)

            doc_id = str(item.get('id', item.get('doc_id', '')))

            content = item.get('contents', item.get('segmented_text', ''))

            if doc_id and content:
                corpus_ids.append(doc_id)

                if isinstance(content, str):
                    tokenized_corpus.append(content.split())
                else:
                    tokenized_corpus.append(content)

    print("   -> Building BM25 Index (Instant)...")
    bm25_model = BM25Okapi(tokenized_corpus)
    print("      ✅ BM25 Ready!")

except Exception as e:
    print(f"❌ Lỗi load Pre-tokenized Data: {e}")
    print("👉 Hãy kiểm tra lại đường dẫn hoặc tên trường (key) trong file JSONL")

print("🚀 Chuyển context sang: improving-learned-index")
%cd /content/improving-learned-index

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    from src.deep_impact.inverted_index import InvertedIndex
    from src.deep_impact.models import DeepImpactXLMR

    print("   -> Loading DeeperImpact Model...")
    deeper_index = InvertedIndex(index_path=INVERTED_INDEX_PATH)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    query_encoder = DeepImpactXLMR.load(CHECKPOINT_PATH)
    query_encoder.to(device)
    query_encoder.eval()
    print("✅ DeeperImpact Loaded!")
except ImportError as e:
    print(f"❌ Lỗi Import DeeperImpact: {e}")

⏳ Đang tải dữ liệu...
   -> Loading Raw Text mapping từ unique_doc_mapping.csv...
      ✅ Loaded 3030 docs for display.
   -> Loading Pre-tokenized Data từ corpus_pretokenized.jsonl...


Reading JSONL: 3030it [00:00, 4784.52it/s]


   -> Building BM25 Index (Instant)...
      ✅ BM25 Ready!
🚀 Chuyển context sang: improving-learned-index
/content/improving-learned-index


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

   -> Loading DeeperImpact Model...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of DeepImpact were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['bert.embeddings.LayerNorm.bias', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.word_embeddings.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.

✅ DeeperImpact Loaded!


In [ ]:
# @title 3. Load ReaderLLM
import importlib.util
import sys
import os
from google.colab import userdata

file_path = "/content/fact-checking-data-framework-vn/src/scripts/reader_llm.py"
module_name = "src.scripts.reader_llm"

print(f"🚀 Loading module trực tiếp từ file: {file_path}")

try:
    spec = importlib.util.spec_from_file_location(module_name, file_path)

    reader_module = importlib.util.module_from_spec(spec)

    sys.modules[module_name] = reader_module

    repo_path = "/content/fact-checking-data-framework-vn"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    spec.loader.exec_module(reader_module)

    ReaderLLM = reader_module.ReaderLLM

    reader_llm = ReaderLLM(api_key=api_key, model_name="gemini-2.5-flash")
    print("✅ ReaderLLM Loaded successfully via importlib!")

except Exception as e:
    print(f"❌ Lỗi: {e}")

🚀 Loading module trực tiếp từ file: /content/fact-checking-data-framework-vn/src/scripts/reader_llm.py
✅ ReaderLLM Loaded successfully via importlib!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# @title 4. Run Demo UI (Updated Interface & Logic)
import gradio as gr
import numpy as np
import torch
from underthesea import word_tokenize

def min_max_normalize(scores_dict):
    """Chuẩn hóa điểm số về khoảng [0, 1]"""
    if not scores_dict: return {}
    values = list(scores_dict.values())
    min_val, max_val = min(values), max(values)
    if max_val == min_val: return {k: 1.0 for k in scores_dict}
    return {k: (v - min_val) / (max_val - min_val) for k, v in scores_dict.items()}

def retrieve_hybrid(query_text, method="Hybrid", top_k=3, alpha=0.5):
    """
    Hàm tìm kiếm kết hợp BM25 và DeeperImpact
    """
    query_bm25 = query_text.lower()
    tokenized_query = word_tokenize(query_bm25)

    bm25_scores = bm25_model.get_scores(tokenized_query)

    # Top 100 candidates
    top_n_bm25 = 100
    top_indices = np.argsort(bm25_scores)[::-1][:top_n_bm25]
    bm25_results = {}
    for idx in top_indices:
        if idx < len(corpus_ids):
            doc_id = str(corpus_ids[idx])
            bm25_results[doc_id] = bm25_scores[idx]

    deeper_results = {}
    if method in ["DeeperImpact", "Hybrid"]:
        with torch.no_grad():
            query_terms = query_encoder.process_query(query_text)

        scored_docs = deeper_index.score(query_terms, top_k=100)
        deeper_results = {str(d): s for d, s in scored_docs}

    final_scores = {}

    if method == "BM25":
        final_scores = bm25_results
    elif method == "DeeperImpact":
        final_scores = deeper_results
    else: # Hybrid
        norm_bm25 = min_max_normalize(bm25_results)
        norm_deeper = min_max_normalize(deeper_results)

        all_keys = set(norm_bm25.keys()) | set(norm_deeper.keys())

        for doc_id in all_keys:
            s_bm25 = norm_bm25.get(doc_id, 0.0)
            s_di = norm_deeper.get(doc_id, 0.0)
            # Weighted Sum
            final_scores[doc_id] = (alpha * s_bm25) + ((1.0 - alpha) * s_di)

    sorted_docs = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    retrieved_docs_for_llm = []
    formatted_context_log = ""

    for rank, (doc_id, score) in enumerate(sorted_docs, 1):
        content = doc_map.get(doc_id, "[Nội dung không tìm thấy trong CSV mapping]")

        retrieved_docs_for_llm.append({
            "id": doc_id,
            "content": content,
            "score": score
        })
        snippet = content[:400] + "..." if len(content) > 400 else content
        formatted_context_log += f"#{rank} [ID: {doc_id}] (Score: {score:.4f})\n{snippet}\n{'-'*30}\n"

    return retrieved_docs_for_llm, formatted_context_log

def pipeline_wrapper(claim, method, alpha_val, top_k_val):
    docs_for_llm, context_log = retrieve_hybrid(claim, method=method, alpha=alpha_val, top_k=int(top_k_val))

    if not docs_for_llm:
        return "## ⚠️ KHÔNG TÌM THẤY THÔNG TIN\nKhông tìm thấy tài liệu nào phù hợp để kiểm chứng.", context_log

    try:
        result = reader_llm.generate_answer(
            qid="demo_query",
            query=claim,
            retrieved_docs=docs_for_llm
        )
        verdict = result.get('verdict', 'Error')
        explanation = result.get('explanation', 'No explanation generated.')

        icon_map = {
            "supported": "✅",
            "refuted": "❌",
            "not enough info": "🤷",
            "nei": "🤷"
        }
        icon = icon_map.get(verdict.lower(), "⚖️")

        final_output = f"## {icon} PHÁN QUYẾT: {verdict.upper()}\n\n**Giải thích chi tiết:**\n{explanation}"

    except Exception as e:
        final_output = f"## ⚠️ LỖI HỆ THỐNG\nĐã xảy ra lỗi khi gọi ReaderLLM: {str(e)}"

    return final_output, context_log


with gr.Blocks(theme=gr.themes.Soft(), title="KLTN Fact-Checking Demo") as demo:
    gr.Markdown("""
    # 🕵️‍♀️ Hệ thống Xác thực Thông tin Tiếng Việt (KLTN Demo)
    **Mô hình:** Hybrid Retrieval (BM25 + DeeperImpact) & Gemini Reader
    """)

    with gr.Row():
        with gr.Column(scale=4):
            inp_claim = gr.Textbox(
                label="Câu nhận định (Claim)",
                placeholder="Nhập thông tin cần kiểm chứng...",
                lines=3
            )
            with gr.Row():
                radio_method = gr.Radio(
                    ["BM25", "DeeperImpact", "Hybrid"],
                    label="Phương pháp Truy hồi",
                    value="Hybrid"
                )
                with gr.Column():
                    slider_alpha = gr.Slider(
                        minimum=0.0, maximum=1.0, value=0.5, step=0.1,
                        label="Alpha (BM25 weight)"
                    )
                    slider_top_k = gr.Slider(
                        minimum=1, maximum=10, value=3, step=1,
                        label="Top K Documents"
                    )

            btn = gr.Button("🔍 Kiểm chứng ngay", variant="primary", size="lg")

            gr.Examples([
                ["Việt Nam tuyên bố chủ quyền đối với hai thực thể địa lý tranh chấp trên Biển Đông là các quần đảo Hoàng Sa (bị mất kiểm soát trên thực tế) và Trường Sa (kiểm soát một phần)."],
                ["Hoa Kỳ là đối tác mậu dịch lớn thứ ba của Singapore."],
                ["Việt Nam là nước xuất khẩu gạo lớn nhất thế giới năm 2023."]
            ], inp_claim)

        with gr.Column(scale=5):
            out_result = gr.Markdown(label="Kết quả Phán quyết")
            out_evidence = gr.TextArea(
                label="Bằng chứng tìm thấy (Top Documents)",
                lines=15,
                interactive=False
            )

    btn.click(
        fn=pipeline_wrapper,
        inputs=[inp_claim, radio_method, slider_alpha, slider_top_k],
        outputs=[out_result, out_evidence]
    )

print("🚀 Đang khởi động Gradio...")
demo.launch(share=True, debug=True)

/tmp/ipython-input-1516310645.py:135: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="KLTN Fact-Checking Demo") as demo:


🚀 Đang khởi động Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://98559b3ba5ec3b6a38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
